This notebook segments speckles, nuclei and FISH spots on different conditions - control TRP, FVP etc. Then calculates the distance of the FISH spot from the nuclear speckles and lamima.

## Importing standard packages and custom fishtools

In [ ]:
import fishtools as ft
import warnings
import numpy as np
import os
import pandas as pd
from aicsimageio import AICSImage
from cellpose import models
import matplotlib.pyplot as plt
from skimage import filters, morphology, measure, feature, segmentation, transform, draw, exposure
import napari

In [ ]:
#Making a list of the files to be analyzed.
czi_list=ft.find_czi_files("/Users/agastya.singh/Desktop/03.09.2026 IF+FISH TRP","Rpl3")

## Human in the loop workflow

### Importing the .czi image and segmenting nuclei, speckles and detecting FISH spots.

In [ ]:
path=czi_list[0] 
print(path)
c1,c2,c3,cell_stage,condition,file=ft.parse_image_properties(path)
gene_647=c1
color_647='yellow'
gene_555=c2
color_555="red"
print(gene_647,gene_555,cell_stage,condition)

# Close if another viewer is open
try:
    if viewer is not None:
        viewer.close()
except Exception:
    pass
viewer=ft.view_napari(path)

# Assign the proper layer order to the data. 0 is the first layer acquired.
layer647=viewer.layers[0].data
layer555=viewer.layers[1].data
specklelayer=viewer.layers[2].data
dapilayer=viewer.layers[3].data
scale,ratio=ft.get_scale(viewer.layers[0])

# Calculating spots and speckles- use this for normal nuclei segmentation 
nuclei_labels,nuclei_mask,num_nuclei_unf=ft.segment_nucleus(dapilayer,scale_zyx=scale,remove_edge_nuclei=True)
# Multiotsu nuclear segmentation for high cytosolic DAPI signals- use this when needed. Usually the otsu thresholding (above) works good enough.
nuclei_labels,nuclei_mask,num_nuclei_unf=ft.segment_nucleus_multiotsu(dapilayer,scale_zyx=scale,remove_edge_nuclei=True, otsu_multiplier=0.7)

# Identify FISH spots on the respective FISH channels. Adjust intensity_threshold_coefficient as needed. Increasing the coefficient removes dim spots. If the FISH signal is not good, use lower coeffiecients.
spots_647,num_spots647_unf=ft.identify_fishspots_soph(layer647,spot_name=gene_647,nuclei_mask=nuclei_mask,spot_color=color_647,scale_zyx=scale,intensity_thresh_coeff=1.2)
spots_555,num_spots555_unf=ft.identify_fishspots_soph(layer555, spot_name=gene_555,nuclei_mask=nuclei_mask,spot_color=color_555,scale_zyx=scale,intensity_thresh_coeff=1.1)

# Control/FVP speckle segmentation
speckle_labels,speckle_mask=ft.segment_speckles(specklelayer,nuclei_mask,scale)
# TRP treated speckle segmentation
speckle_labels,speckle_mask=ft.segment_speckles_mulitotsu(specklelayer,nuclei_mask,scale,otsu_multiplier=1) # As the speckle architecture and distribution changes, canonical speckle segmentation code using single otsu thresholding is not optimal.

# Adding to napari viewer.
viewer.add_labels(nuclei_labels,scale=scale,opacity=0.3,blending="additive")
viewer.add_points(spots_647,scale=scale,opacity=100,blending="additive",face_color='transparent',border_color=color_647, border_width=0.1)
viewer.add_points(spots_555,scale=scale,opacity=100,blending="additive",face_color='transparent',border_color=color_555, border_width=0.1)
viewer.add_labels(speckle_labels,scale=scale,opacity=0.3,blending="additive", visible=False)

# After this, manual check of the points is done in napari GUI. Points are added or removed as needed.

Now a napari viewer GUI starts where you can check the segmentation and spot calling. Use eyes to ensure proper spot calling and segmentation occurs. If not go back to the previous steps and optimize parameters.

### Update the FISH spots after manual check

In [ ]:
# Update coordinates of points after manual check.
print(f"Old spots_647: {len(spots_647)}\nOld spots_555: {len(spots_555)}")

# Refreshing 555 spots
layer_name="spots_555"
layer = viewer.layers[layer_name]
spots_555 = layer.data

# Refreshing 647 spots
layer_name="spots_647"
layer = viewer.layers[layer_name]
spots_647 = layer.data

print(f"New spots_647: {len(spots_647)}\nNew spots_555: {len(spots_555)}")

# Saving the point coordinates per file for positive training data... This is just for lab use. We want to train a ML model to call spots. Not needed for this analysis.
df_647 = pd.DataFrame(spots_647, columns=["z", "y", "x"])
df_647["gene"] = gene_647
df_555 = pd.DataFrame(spots_555, columns=["z", "y", "x"])
df_555["gene"] = gene_555

spots_df = pd.concat([df_647, df_555], ignore_index=True)
# spots_df = df_647 

spots_df.to_csv(os.path.splitext(path)[0] + "_spots_data_manualcheck_v1.tsv", sep="\t", index=False)


### Computing distances of FISH spots from the nearest nuclear speckles
Compute Euclidean Distance Transform on the nuclear mask to calculate distances from nuclear lamina.
Compute EDT of the segmented speckles inside the chromatin mask. Chromatin mask excludes low DAPI regions like the Nucleolar Precursor Bodies as there is no DNA.
Now, overlay the FISH spots on the lamina EDT and speckle EDT- this gives distance of the spot from lamina and speckles.
Summarise the results in a table.

In [ ]:
# Calculate masks and distances and save.
chromatin_mask=ft.mask_chromatin(dapilayer,scale)
lamina_edt=ft.perform_edt(nuclei_mask,scale)[0]
print("Lamina EDT done")
speckles_edt=ft.calculate_euclidean_distance_from_speckles(speckle_mask,nuclei_mask,chromatin_mask,scale)
# Summarising the data
summary_df=ft.summary_table(spots_647,nuclei_labels,speckle_labels,lamina_edt,speckles_edt,scale,path=path,centroids2=spots_555,centroids1_name=gene_647,centroids2_name=gene_555)
# Saving the output dataframe.
save_path_output=os.path.splitext(path)[0]+"_summarydf_manualcheck_v2.tsv"
summary_df.to_csv(save_path_output,sep="\t",header=True,index=True)
print(os.path.basename(path))
summary_df